In [187]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [188]:
with open('names.txt','r') as f:
    words = f.read().splitlines()

In [189]:
characters = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(characters)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [190]:
import torch

In [191]:
import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

train_words = words[:n1]   
dev_words   = words[n1:n2] 
test_words  = words[n2:] 


In [192]:
N= torch.zeros((27,27,27), dtype=torch.int32)

In [193]:
import numpy as np

ks=[]
lss=[]
N = torch.zeros((27, 27, 27), dtype=torch.int32)
for w in train_words:
    chrs = ['.', '.'] + list(w) + ['.']
    for ch1, ch2, ch3 in zip(chrs, chrs[1:], chrs[2:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        idx3 = stoi[ch3]
        N[idx1,idx2,idx3] +=1

for k in np.arange(0.25, 2.25, 0.25).tolist():
    P = (N + k).float()
    P /= P.sum(dim=2, keepdim=True)

    log_likelihood = 0.0
    n = 0
    for w in dev_words:
        chrs = ['.', '.'] + list(w) + ['.']
        for ch1, ch2, ch3 in zip(chrs, chrs[1:], chrs[2:]):
            prob = P[stoi[ch1], stoi[ch2], stoi[ch3]]
            log_likelihood += torch.log(prob)
            n += 1
            
    dev_loss = -log_likelihood / n
    ks.append(k)
    lss.append(dev_loss)
    print(f'Smoothing (k={k:4.2f}) -> Dev Loss: {dev_loss.item():.4f}')


Smoothing (k=0.00) -> Dev Loss: nan
Smoothing (k=0.25) -> Dev Loss: 2.2227
Smoothing (k=0.50) -> Dev Loss: 2.2267
Smoothing (k=0.75) -> Dev Loss: 2.2316
Smoothing (k=1.00) -> Dev Loss: 2.2365
Smoothing (k=1.25) -> Dev Loss: 2.2413
Smoothing (k=1.50) -> Dev Loss: 2.2461
Smoothing (k=1.75) -> Dev Loss: 2.2508


In [194]:
min_loss = min(lss)
best_k = ks[lss.index(min_loss)]
p = (N + best_k).float()
p /= p.sum(2, keepdim=True)


In [195]:
log_likelihood = 0.0
n = 0
for w in test_words:
    chrs = ['.','.'] + list(w) + ['.']
    for ch1, ch2,ch3 in zip(chrs, chrs[1:],chrs[2:]):
        idx1 = stoi[ch1]
        idx2 = stoi[ch2]
        idx3 = stoi[ch3]
        prob = p[idx1,idx2,idx3]
        log_prob = torch.log(prob)
        log_likelihood += log_prob
        n+=1

test_loss = -log_likelihood / n
print(f"Trigram Test Loss: {test_loss.item():.4f}")
print("Bigram Test Loss : ~2.454")
print(f"Fark: Trigram modeli ~{2.454 - test_loss.item():.4f} puan daha başarılı!")


Trigram Test Loss: nan
Bigram Test Loss : ~2.454
Fark: Trigram modeli ~nan puan daha başarılı!


In [196]:
generator = torch.Generator().manual_seed(42)
print("--- Trigram Sayım Modeli ile Üretilen İsimler ---")
for i in range(5):
    out = []
    idx1, idx2 = 0, 0
    while True:
        p_ = p[idx1, idx2]
        idx3 = torch.multinomial(p_, num_samples=1, replacement=True, generator=generator).item()
        out.append(itos[idx3])
        if idx3 == 0:
            break
        idx1, idx2 = idx2, idx3
    print("".join(out))


--- Trigram Sayım Modeli ile Üretilen İsimler ---
ye.
syahle.
amen.
leekkim.
mannya.


In [197]:
import torch.nn.functional as F

In [198]:
W = torch.randn((54, 27), generator=generator, requires_grad=True)

In [199]:
# dataesti oluştur
xs1,xs2, ys = [], [],[]

for w in words:
  chrs = ['.','.'] + list(w) + ['.']
  for ch1, ch2,ch3 in zip(chrs, chrs[1:],chrs[2:]):
      idx1 = stoi[ch1]
      idx2 = stoi[ch2]
      idx3 = stoi[ch3]
      xs1.append(idx1)
      xs2.append(idx2)
      ys.append(idx3)

xs1 = torch.tensor(xs1)
xs2 = torch.tensor(xs2)
ys = torch.tensor(ys)

num = xs1.nelement() + xs2.nelement()
print('number of examples: ', num)

number of examples:  456292


In [200]:
# gradient descent
W = torch.randn((54, 27), generator=generator, requires_grad=True)
for k in range(100):
  
  # forward pass
  xenc1 = F.one_hot(xs1, num_classes=27).float()
  xenc2 = F.one_hot(xs2, num_classes=27).float()
  xenc = torch.cat([xenc1, xenc2], dim=1)  
  logits = xenc @ W 
  counts = logits.exp() #softmax
  probs = counts / counts.sum(dim=1, keepdims=True) #softmax
  loss = -probs[torch.arange(len(ys)), ys].log().mean() + 0.01*(W**2).mean()#nll loss  
  
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -10 * W.grad

4.242650985717773
4.033442974090576
3.866694927215576
3.732417106628418
3.6237125396728516
3.532491445541382
3.453294515609741
3.383469820022583
3.321467161178589
3.266094446182251
3.2163636684417725
3.1714529991149902
3.1306850910186768
3.09350848197937
3.059471607208252
3.0282044410705566
2.9994025230407715
2.972809076309204
2.948207378387451
2.925408363342285
2.9042468070983887
2.8845739364624023
2.866257667541504
2.8491764068603516
2.833220958709717
2.8182897567749023
2.8042914867401123
2.791142702102661
2.778768539428711
2.767099380493164
2.7560739517211914
2.7456369400024414
2.7357378005981445
2.726332664489746
2.7173805236816406
2.7088463306427
2.700697422027588
2.692905902862549
2.6854467391967773
2.678295612335205
2.6714320182800293
2.6648380756378174
2.658496856689453
2.652393102645874
2.646512746810913
2.640842914581299
2.6353719234466553
2.630089282989502
2.6249852180480957
2.6200499534606934
2.6152756214141846
2.610653877258301
2.60617733001709
2.601839303970337
2.59763312

In [202]:
for i in range(5):
  out = []
  idx1 = 0
  idx2 = 0
  while True:
    xenc1 = F.one_hot(torch.tensor([idx1]), num_classes=27).float()
    xenc2 = F.one_hot(torch.tensor([idx2]), num_classes=27).float()
    xenc = torch.cat([xenc1, xenc2], dim=1)  
    logits = xenc @ W
    counts = logits.exp() 
    p = counts / counts.sum(1, keepdims=True)
    # ----------
    
    idx3= torch.multinomial(p, num_samples=1, replacement=True, generator=generator).item()
    out.append(itos[idx3])
    if idx3 == 0:
      break
    idx1 = idx2
    idx2 = idx3
  print(''.join(out))

mmiray.
na.
tea.
jaa.
sines.
